# SigLIP2 美学头 — AVA 真标签直训 (no distillation)

复刻 LAION `improved-aesthetic-predictor` 的造法，但 backbone 用你的 **frozen SigLIP2**，标签用公开 **AVA**(真人 1–10 评分) 而非 teacher 蒸馏。

对比蒸馏版的改进（解决分数压不开问题）：
1. **真人标注**，无 teacher 偏置；AVA 有真实高/低分尾部。
2. **balanced MSE**：AVA 原始分布在 ~5 处极峰，直接 MSE 仍会向均值收缩；按分数分箱反频率加权，强制头平等看到两端 → 分数拉得开。
3. dropout 调小(0.1)、weight_decay→0，进一步减少收缩。

**架构与导出和蒸馏版完全一致**（L2norm 内置 + 同样 MLP），所以 lumen-hub / lumen-convert 不用改，只换 bpk 权重。

用法：选 GPU runtime → 设 `STUDENT_ID` → Run all → 换另一个 backbone 再跑一次。

## 1. 安装依赖（无需 teacher）

In [ ]:
# pillow 钉到 11.2.1：Colab 默认 pillow 被 -U 升级后 ImageText/_Ink 文件不一致会炸。
%pip install -q -U "transformers>=4.49" accelerate safetensors datasets scipy onnx onnxruntime tqdm
%pip install -q "pillow==11.2.1"
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select a GPU runtime (A100/L4).'

STUDENT_ID = "google/siglip2-so400m-patch14-384"   # 或 "google/siglip2-base-patch16-224"

# AVA 数据集。默认用已确认 schema(image + mean_score)、min50票、按分数分10档分层的子集——
# 干净、已平衡、25.5k 足够训头。想要更多数据可换 "trojblue/AVA-Huggingface"(全量, 列名先自查)。
AVA_DATASET = "trojblue/AVA-aesthetics-10pct-min50-10bins"
AVA_SPLIT   = "train"
SCORE_COL   = "mean_score"                          # 标签列（1–10）
IMAGE_COL   = "image"
N_IMAGES    = 60000                                 # 子集 ~20k 训练，全量 AVA ~25w

BALANCE     = True                                  # 按分数分箱反频率加权(治压缩的关键)
N_BINS      = 10
BATCH       = 64
EPOCHS      = 80
HEAD_LR     = 1e-3
DROPOUT     = 0.1                                    # 比蒸馏版(0.3)小，减少收缩
WEIGHT_DECAY = 0.0
VAL_FRAC    = 0.1
SEED        = 42

WORK = "/content/aesthetic_ava"
import os; os.makedirs(WORK, exist_ok=True)
import random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda"; SHORT = STUDENT_ID.split('/')[-1]
print('student =', STUDENT_ID, '| dataset =', AVA_DATASET)

In [ ]:
STUDENT_ID = "google/siglip2-so400m-patch14-384"   # 或 "google/siglip2-base-patch16-224"

# AVA 数据集：全量更多样，10pct-min50-10bins 已分层、票数更可靠且更小更快。
AVA_DATASET = "trojblue/AVA-Huggingface"            # 或 "trojblue/AVA-aesthetics-10pct-min50-10bins"
AVA_SPLIT   = "train"
SCORE_COL   = "mean_score"                          # 标签列（1–10）
IMAGE_COL   = "image"
N_IMAGES    = 60000                                 # 取多少张（全量 AVA ~25w，子集更少）

BALANCE     = True                                  # 按分数分箱反频率加权(治压缩的关键)
N_BINS      = 10
BATCH       = 64
EPOCHS      = 80
HEAD_LR     = 1e-3
DROPOUT     = 0.1                                    # 比蒸馏版(0.3)小，减少收缩
WEIGHT_DECAY = 0.0
VAL_FRAC    = 0.1
SEED        = 42

WORK = "/content/aesthetic_ava"
import os; os.makedirs(WORK, exist_ok=True)
import random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda"; SHORT = STUDENT_ID.split('/')[-1]
print('student =', STUDENT_ID, '| dataset =', AVA_DATASET)

## 3. Frozen SigLIP2 student（取 pooler_output = 你 runtime 的 image_features）

In [ ]:
from transformers import AutoModel, AutoProcessor

student = AutoModel.from_pretrained(STUDENT_ID).to(DEVICE).eval()
student_proc = AutoProcessor.from_pretrained(STUDENT_ID)
for p in student.parameters():
    p.requires_grad_(False)
HIDDEN = student.config.vision_config.hidden_size
print('image_features dim =', HIDDEN)

@torch.inference_mode()
def student_embed(pil_batch):
    inp = student_proc(images=pil_batch, return_tensors="pt")
    vis = {k: v.to(DEVICE) for k, v in inp.items() if k.startswith('pixel') or k in ('spatial_shapes',)}
    return student.vision_model(**vis).pooler_output.float().cpu()

## 4. 流式 AVA，预计算 (embedding, mean_score)，带缓存

In [ ]:
from PIL import Image, ImageFile
from datasets import load_dataset, Image as HFImage
from tqdm.auto import tqdm
import itertools, io
Image.MAX_IMAGE_PIXELS = None
ImageFile.LOAD_TRUNCATED_IMAGES = True          # 截断图也能读，不抛异常

def _to_pil(img):
    if isinstance(img, dict):
        if img.get('bytes'):  img = Image.open(io.BytesIO(img['bytes']))
        elif img.get('path'): img = Image.open(img['path'])
        else: return None
    return img.convert('RGB')

def ava_iter():
    # 非流式：下载并缓存一次(~3.4GB)，两个 backbone 复用、不重下。想流式改 streaming=True。
    ds = load_dataset(AVA_DATASET, split=AVA_SPLIT)
    ds = ds.cast_column(IMAGE_COL, HFImage(decode=False))   # 取原始 bytes，自己解码以便跳过坏图
    for ex in ds:
        s = ex.get(SCORE_COL)
        img = ex.get(IMAGE_COL)
        if s is None or img is None:
            continue
        try:
            pil = _to_pil(img)
            if pil is not None:
                yield pil, float(s)
        except Exception:
            continue                                        # 真损坏的图直接跳过

cache_path = os.path.join(WORK, f'ava_{SHORT}.pt')
if os.path.exists(cache_path):
    blob = torch.load(cache_path); EMB, SCORE = blob['emb'], blob['score']
    print('loaded cache', EMB.shape)
else:
    embs, scores, done = [], [], 0
    pbar = tqdm(total=N_IMAGES)
    it = ava_iter()
    while done < N_IMAGES:
        chunk = list(itertools.islice(it, BATCH))
        if not chunk: break
        imgs = [c[0] for c in chunk]; ys = [c[1] for c in chunk]
        try:
            e = student_embed(imgs)
        except Exception as ex:
            print('skip batch:', ex); continue
        embs.append(e); scores.append(torch.tensor(ys))
        done += len(chunk); pbar.update(len(chunk))
    pbar.close()
    EMB = torch.cat(embs)[:N_IMAGES]; SCORE = torch.cat(scores)[:N_IMAGES]
    torch.save({'emb': EMB, 'score': SCORE}, cache_path)
    print('cached', EMB.shape, '->', cache_path)

print('AVA score range %.2f .. %.2f (mean %.2f, std %.2f)' % (SCORE.min(), SCORE.max(), SCORE.mean(), SCORE.std()))

## 5. Head（与蒸馏版/lumen-hub 架构完全一致）+ balanced 训练

In [ ]:
import torch.nn as nn
from scipy.stats import spearmanr, pearsonr

class AestheticHead(nn.Module):
    """输入 = 原始 SigLIP2 image_features (pooler_output)。内部 L2 归一化后过 MLP -> 标量分。
       架构与蒸馏版一致；dropout 仅训练期生效，导出 ONNX 时折掉，不改 burn arch。"""
    def __init__(self, in_dim, p=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 1024), nn.ReLU(), nn.Dropout(p),
            nn.Linear(1024, 128),    nn.ReLU(), nn.Dropout(p),
            nn.Linear(128, 64),      nn.ReLU(), nn.Dropout(p),
            nn.Linear(64, 16),       nn.ReLU(),
            nn.Linear(16, 1),
        )
    def forward(self, image_features):
        x = image_features / image_features.norm(dim=-1, keepdim=True).clamp_min(1e-6)
        return self.net(x).squeeze(-1)

# ── balanced 样本权重：按分数分箱取反频率，强制头平等看到两端 ──
if BALANCE:
    bins = torch.linspace(SCORE.min(), SCORE.max() + 1e-4, N_BINS + 1)
    idx = torch.bucketize(SCORE, bins) - 1
    idx = idx.clamp(0, N_BINS - 1)
    counts = torch.bincount(idx, minlength=N_BINS).float()
    w_per_bin = (counts.sum() / (counts + 1e-6))
    w_per_bin = w_per_bin / w_per_bin.mean()        # 归一，均值=1
    W = w_per_bin[idx]
    print('bin counts:', counts.int().tolist())
else:
    W = torch.ones_like(SCORE)

g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(EMB), generator=g)
n_val = int(len(EMB) * VAL_FRAC)
val_idx, tr_idx = perm[:n_val], perm[n_val:]
Xtr, Ytr, Wtr = EMB[tr_idx].to(DEVICE), SCORE[tr_idx].to(DEVICE), W[tr_idx].to(DEVICE)
Xva, Yva = EMB[val_idx].to(DEVICE), SCORE[val_idx].to(DEVICE)

head = AestheticHead(HIDDEN, p=DROPOUT).to(DEVICE)
opt = torch.optim.AdamW(head.parameters(), lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
bs = 4096
best_srcc, best_state = -1.0, None

for ep in range(EPOCHS):
    head.train()
    order = torch.randperm(len(Xtr), device=DEVICE)
    for i in range(0, len(order), bs):
        b = order[i:i+bs]
        opt.zero_grad()
        pred = head(Xtr[b])
        loss = (Wtr[b] * (pred - Ytr[b]) ** 2).mean()   # weighted MSE
        loss.backward(); opt.step()
    sched.step()
    head.eval()
    with torch.inference_mode():
        pv = head(Xva).cpu().numpy()
    gt = Yva.cpu().numpy()
    srcc = spearmanr(pv, gt).statistic; plcc = pearsonr(pv, gt)[0]
    if srcc > best_srcc:
        best_srcc = srcc
        best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
    if ep % 5 == 0 or ep == EPOCHS - 1:
        print(f'ep {ep:02d}  SRCC {srcc:.4f}  PLCC {plcc:.4f}  pred[{pv.min():.2f},{pv.max():.2f}]  gt[{gt.min():.2f},{gt.max():.2f}]')

head.load_state_dict(best_state)
print(f'\nBEST val SRCC = {best_srcc:.4f}')

## 6. 范围对比（确认分数拉开了）

In [ ]:
head.eval()
with torch.inference_mode():
    pv = head(Xva).cpu().numpy()
gt = Yva.cpu().numpy()
import numpy as np
for q in (1, 10, 50, 90, 99):
    print(f'p{q:02d}: pred {np.percentile(pv,q):5.2f}   label {np.percentile(gt,q):5.2f}')
print(f'pred std {pv.std():.2f} vs label std {gt.std():.2f}  (越接近说明刻度越没被压)')

## 7. 导出 ONNX + safetensors（与蒸馏版完全一致 → 直接喂 lumen-convert）

In [ ]:
import json, onnxruntime as ort
from safetensors.torch import save_file

out_dir = os.path.join(WORK, f'aesthetic-head-{SHORT}')
os.makedirs(out_dir, exist_ok=True)
head.eval().cpu()
save_file(head.state_dict(), os.path.join(out_dir, 'head.safetensors'))

config = {
    'backbone': STUDENT_ID, 'input': 'image_features (SigLIP2 vision pooler_output)',
    'in_dim': HIDDEN, 'l2_normalize_input': True, 'mlp': [HIDDEN, 1024, 128, 64, 16, 1],
    'labels': f'AVA mean_score via {AVA_DATASET}', 'balanced': BALANCE,
    'score_range': '~1..10 (AVA scale)', 'val_srcc': round(float(best_srcc), 4), 'n_images': int(len(EMB)),
}
with open(os.path.join(out_dir, 'config.json'), 'w') as f:
    json.dump(config, f, indent=2)

dummy = torch.randn(1, HIDDEN)
torch.onnx.export(head, dummy, os.path.join(out_dir, 'aesthetic.onnx'),
    input_names=['image_features'], output_names=['aesthetic_score'],
    dynamic_axes={'image_features': {0: 'batch'}, 'aesthetic_score': {0: 'batch'}}, opset_version=17)

sess = ort.InferenceSession(os.path.join(out_dir, 'aesthetic.onnx'))
ox = sess.run(None, {'image_features': EMB[:8].numpy()})[0]
with torch.inference_mode():
    pt = head(EMB[:8]).numpy()
print('onnx vs pt max abs diff =', float(abs(ox - pt).max()))
print('exported ->', out_dir); print(json.dumps(config, indent=2))

In [ ]:
import shutil
zip_path = shutil.make_archive(out_dir, 'zip', out_dir)
print('zip ->', zip_path)
try:
    from google.colab import files; files.download(zip_path)
except Exception:
    pass

## 接入

导出的 `aesthetic.onnx` 与蒸馏版结构完全相同（`ReduceL2/Clip/Div/Gemm/Relu/Squeeze`，输入 `image_features`），所以直接走原流程：

1. 放到 `crates/lumen-convert/onnx/aesthetic-head-siglip2-<arch>/aesthetic.onnx`（覆盖旧的）。
2. `uv run tools/siglip/onnx_prep.py ... aesthetic.prepared.onnx --opset 21`。
3. `cargo build -p lumen-convert` → `cargo run --release -p lumen-convert --bin export_aesthetic_head`。
4. 得到新的 `aesthetic.{fp32,fp16,fp16q8}.bpk`，覆盖 lumen-models / 缓存即可。**lumen-hub / SDK 不用动。**

两个 backbone 各跑一次（换 `STUDENT_ID`）。